# Notebook 01 — SMAP data exploration

*Corrected version of the original `01_dataExploration.ipynb`.*

**What this notebook does.** It looks at the NASA SMAP telemetry release before any
modelling: how many channels there are, their shapes, their value ranges, how much of each
test file is labelled anomalous, and how many normal windows each channel can provide.

**Why.** Every later notebook depends on these facts. The original exploration got several
of them wrong, and some of its conclusions were carried forward.

**Input.** `labeled_anomalies.csv` and the `train/` and `test/` folders of the telemanom
release. Found automatically (local repository folder `SMAP - NASA`, or `/kaggle/input`).

**Output.** Two figures and `exploration_summary.json` in the output folder.

### What was corrected

1. **P-2 is listed twice** in `labeled_anomalies.csv`, with two different anomaly ranges.
   The original counted 82 channels and 55 SMAP channels. There are 81 unique channels and
   **54 unique SMAP channels**. The two P-2 rows are now merged.
2. **"Already normalised: Yes" was wrong.** The original checked only the maximum of one
   channel (P-1, range -1 to 1). Across all channels the values run from about -1.5 to 258.
   All channels are now checked, minimum and maximum.
3. **Anomaly ranges are inclusive** at both ends. The original mask dropped the last
   timestep of every range.
4. **Shot feasibility was measured on the wrong thing.** The original counted anomaly
   *sequences* per channel and concluded that no channel supports 5-shot. The final design
   adapts on K **normal** windows, so feasibility is now measured by normal windows.
5. The claim that 12% anomalies "proves" supervised learning is impossible was removed;
   the numbers are reported without that interpretation.
6. Paths no longer depend on `os.chdir('..')` and a Windows folder layout.

In [1]:
import os, sys

def _find_common():
    """Find maml_common.py: this folder when run locally, /kaggle/input on Kaggle."""
    for root in [os.getcwd(), "/kaggle/input"]:
        if os.path.isdir(root):
            for d, _, files in os.walk(root):
                if "maml_common.py" in files:
                    return d
    raise FileNotFoundError("maml_common.py not found. Run from the Corrected-Notebooks folder, "
                            "or attach that folder to Kaggle as a Dataset.")

sys.path.insert(0, _find_common())
import maml_common as sc
SMOKE = os.environ.get("SMAP_SMOKE") == "1"     # tiny settings for testing only
OUT = sc.output_dir(smoke=SMOKE)
print("shared code:", sc.__file__)
print("outputs go to:", OUT)
print("code version:", sc.git_commit())

import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 140)

shared code: /home/user/Objective-2/Corrected-Notebooks/maml_common.py
outputs go to: /home/user/Objective-2/Corrected-Notebooks/outputs
code version: 85c9644f9e8f5ee6eae4e65bf032fc8577223ed3


## 1 — The labels file

Each row describes one channel: its spacecraft (SMAP or MSL) and the anomalous ranges in
its test file. We first read the raw rows, look for channels listed more than once, and
then build one merged row per channel.

In [2]:
csv, TRAIN_DIR, TEST_DIR = sc.find_smap_raw()
raw = pd.read_csv(csv)
print("rows in the CSV:", len(raw))
print(raw["spacecraft"].value_counts().to_string())
dups = raw[raw["chan_id"].duplicated(keep=False)]
print("\nchannels listed more than once:")
print(dups[["chan_id", "spacecraft", "anomaly_sequences", "num_values"]].to_string(index=False))

labels = sc.load_labels(csv)
print("\nunique channels:", len(labels))
print(labels["spacecraft"].value_counts().to_string())
print("\nmerged P-2 ranges:", labels.loc["P-2", "sequences"])

rows in the CSV: 82
spacecraft
SMAP    55
MSL     27

channels listed more than once:
chan_id spacecraft anomaly_sequences  num_values
    P-2       SMAP    [[5350, 6575]]        8209
    P-2       SMAP    [[5300, 6420]]        8209

unique channels: 81
spacecraft
SMAP    54
MSL     27

merged P-2 ranges: [[5350, 6575], [5300, 6420]]


## 2 — Shapes and number of features

SMAP channels should all have 25 features; MSL channels have 55. The model in this project
takes 25 features, so only SMAP channels are used later.

In [3]:
rows = []
for ch in labels.index:
    tr = np.load(os.path.join(TRAIN_DIR, f"{ch}.npy"))
    te = np.load(os.path.join(TEST_DIR, f"{ch}.npy"))
    rows.append({"channel": ch, "spacecraft": labels.loc[ch, "spacecraft"],
                 "train_len": tr.shape[0], "test_len": te.shape[0], "n_features": tr.shape[1],
                 "train_min": tr.min(), "train_max": tr.max(), "test_min": te.min(), "test_max": te.max()})
stats = pd.DataFrame(rows).set_index("channel")
print(stats.groupby("spacecraft").agg(channels=("train_len", "size"),
                                      features=("n_features", lambda s: sorted(set(s))),
                                      mean_train_len=("train_len", "mean"),
                                      mean_test_len=("test_len", "mean")).round(0).to_string())
assert (stats.loc[stats.spacecraft == "SMAP", "n_features"] == 25).all()
assert (stats.spacecraft == "SMAP").sum() == 54

            channels features  mean_train_len  mean_test_len
spacecraft                                                  
MSL               27     [55]          2160.0         2731.0
SMAP              54     [25]          2556.0         8071.0


## 3 — Are the values already scaled to [0, 1]?

The original looked at one channel and concluded yes. Here every channel is checked.

In [4]:
lo = stats[["train_min", "test_min"]].min(axis=1)
hi = stats[["train_max", "test_max"]].max(axis=1)
outside = stats[(lo < 0) | (hi > 1)]
print(f"overall range: {lo.min():.4f} to {hi.max():.4f}")
print(f"channels with values outside [0, 1]: {len(outside)} of {len(stats)} "
      f"(SMAP: {(outside.spacecraft == 'SMAP').sum()})")
print("channels with the largest maximum:")
print(hi.sort_values(ascending=False).head(5).round(3).to_string())
print("\nConclusion from the data:", "values are NOT all in [0, 1]; scaling is needed"
      if len(outside) else "all values are in [0, 1]")

overall range: -1.4772 to 258.1081
channels with values outside [0, 1]: 81 of 81 (SMAP: 54)
channels with the largest maximum:
channel
M-6    258.108
F-5      6.954
M-1      2.492
C-1      2.193
F-4      1.465

Conclusion from the data: values are NOT all in [0, 1]; scaling is needed


## 4 — Two example channels

The blue line is the first feature of the test file; red bands are the labelled anomalies.
P-1 is the channel the original plotted; D-3 is one of the evaluation channels.

In [5]:
def plot_channel(ch, path):
    te = np.load(os.path.join(TEST_DIR, f"{ch}.npy"))
    fig, ax = plt.subplots(figsize=(14, 3.5))
    ax.plot(te[:, 0], lw=0.8, color="steelblue", label="feature 0")
    for s, e in labels.loc[ch, "sequences"]:
        ax.axvspan(s, e + 1, color="red", alpha=0.25)
    ax.set_title(f"{ch}: test file, feature 0, labelled anomalies shaded")
    ax.set_xlabel("timestep"); ax.set_ylabel("raw value")
    fig.tight_layout(); fig.savefig(path, dpi=120); plt.close(fig)
    print("saved", path)

for ch in ["P-1", "D-3"]:
    plot_channel(ch, os.path.join(OUT, f"nb01_channel_{ch}.png"))

saved /home/user/Objective-2/Corrected-Notebooks/outputs/nb01_channel_P-1.png
saved /home/user/Objective-2/Corrected-Notebooks/outputs/nb01_channel_D-3.png


## 5 — How much of each test file is anomalous

Labels are built per timestep with inclusive range ends. We report the pooled fraction
(all SMAP test timesteps together) and the mean of per-channel fractions, because they
differ. For the seven evaluation channels we also note whether the anomaly runs to the end
of the file, which gives those channels a high anomaly fraction.

In [6]:
prev = []
for ch in labels.index:
    n = int(stats.loc[ch, "test_len"])
    y = sc.point_labels(labels.loc[ch, "sequences"], n)
    last_end = max(e for _, e in labels.loc[ch, "sequences"])
    prev.append({"channel": ch, "spacecraft": labels.loc[ch, "spacecraft"], "test_len": n,
                 "anomalous_points": int(y.sum()), "fraction": y.mean(),
                 "anomaly_reaches_end": last_end >= n - 1})
prev = pd.DataFrame(prev).set_index("channel")
for sp, g in prev.groupby("spacecraft"):
    print(f"{sp}: pooled anomaly fraction {g.anomalous_points.sum() / g.test_len.sum():.4f}, "
          f"mean of per-channel fractions {g.fraction.mean():.4f}")
print("\nevaluation channels (and P-4, reported separately):")
print(prev.loc[sc.EVAL_CHANNELS + sc.SENSITIVITY_CHANNELS,
               ["test_len", "anomalous_points", "fraction", "anomaly_reaches_end"]].round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(prev.loc[prev.spacecraft == "SMAP", "fraction"], bins=20, color="steelblue", edgecolor="white")
ax.set_xlabel("anomalous fraction of the test file"); ax.set_ylabel("SMAP channels")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "nb01_anomaly_fraction.png"), dpi=120); plt.close(fig)

MSL: pooled anomaly fraction 0.1053, mean of per-channel fractions 0.1202
SMAP: pooled anomaly fraction 0.1284, mean of per-channel fractions 0.1244

evaluation channels (and P-4, reported separately):
         test_len  anomalous_points  fraction  anomaly_reaches_end
channel                                                           
E-3          8307              3213     0.387                 True
D-7          7642              2702     0.354                 True
E-6          8300                66     0.008                False
D-6          7884                81     0.010                False
T-2          8625              1785     0.207                 True
A-6          4453                41     0.009                False
D-3          8640              3276     0.379                False
P-4          7783               443     0.057                False


## 6 — What the few-shot design actually needs

The model adapts on K **normal** windows (length 30) taken from the channel's training
file, with K = 1, 5 or 10. Anomalies are only used for scoring. So the question is how many
normal windows each channel has. Windows are cut every 10 timesteps, as in the original.
Channels with fewer than 40 normal windows cannot give a 10-shot support set and still
keep normal data aside.

In [7]:
smap = stats[stats.spacecraft == "SMAP"].copy()
smap["normal_windows_stride10"] = (smap.train_len - sc.WINDOW) // sc.STRIDE_NORMAL + 1
print(smap["normal_windows_stride10"].describe().round(1).to_string())
print("\nSMAP channels with fewer than 40 normal windows:")
print(smap.loc[smap.normal_windows_stride10 < 40, ["train_len", "normal_windows_stride10"]].to_string())

count     54.0
mean     253.3
std       65.6
min       29.0
25%      257.2
50%      282.5
75%      286.0
max      286.0

SMAP channels with fewer than 40 normal windows:
         train_len  normal_windows_stride10
channel                                    
D-12           312                       29


## 7 — Save the summary

In [8]:
summary = {
    "csv_rows": int(len(raw)), "unique_channels": int(len(labels)),
    "unique_smap_channels": int((labels.spacecraft == "SMAP").sum()),
    "duplicate_rows": dups["chan_id"].tolist(),
    "value_range_all_channels": [float(lo.min()), float(hi.max())],
    "channels_outside_0_1": int(len(outside)),
    "smap_pooled_anomaly_fraction": float(prev[prev.spacecraft == "SMAP"].anomalous_points.sum()
                                          / prev[prev.spacecraft == "SMAP"].test_len.sum()),
    "smap_mean_channel_anomaly_fraction": float(prev[prev.spacecraft == "SMAP"].fraction.mean()),
    "eval_channels": {c: {"test_fraction": float(prev.loc[c, "fraction"]),
                          "anomaly_reaches_end": bool(prev.loc[c, "anomaly_reaches_end"]),
                          "normal_windows_stride10": int(smap.loc[c, "normal_windows_stride10"])}
                      for c in sc.EVAL_CHANNELS + sc.SENSITIVITY_CHANNELS},
    "smap_channels_under_40_normal_windows": smap.index[smap.normal_windows_stride10 < 40].tolist(),
}
print(sc.save_json(os.path.join(OUT, "exploration_summary.json"), summary))

/home/user/Objective-2/Corrected-Notebooks/outputs/exploration_summary.json
